#Feature Customer Product

##Data Definitions

In [0]:
CATALOG_NAME = "ml_training_dev"
SOURCE_SCHEMA_NAME = "gold"
TARGET_SCHEMA_NAME = "feature"
SOURCE_TABLE_NAME = "customers_orders_category"
TARGET_TABLE_NAME = "customer_product_interactions"

##Create Catalog and schema

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS {}.{}".format(CATALOG_NAME, TARGET_SCHEMA_NAME))

##Import Libraries

In [0]:
from pyspark.sql.functions import current_date, count,sum, avg, round, max, to_date,col, min, countDistinct


##Reading Source Tables

In [0]:
df_feature_table = spark.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}").drop("loadDate")

##Transformations

In [0]:
df_final = (df_feature_table
            .groupBy("customer_unique_id",
                    "product_id")
            .agg(count("product_id").alias("purchases"),
                 countDistinct("product_id").alias("unique_orders"),
                 round(sum("price"),2).alias("total_spent"),
                 round(avg("price"), 2).alias("avg_price"),
                 round(avg("freight_value"), 2).alias("avg_freight"),
                 max(to_date(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss")).alias("last_purchase"),
                 min(to_date(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss")).alias("first_purchase")
                 
                 )
            )

##Writting table

In [0]:
df_final.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}")
print("table successfully created!")

In [0]:
#df_final.orderBy(col("purchases").desc()).display()
# label = 1
# 2018-08-01 → 2018-08-31

In [0]:
# purchase_count
# total_spent
# avg_price
# avg_freight
# first_purchase
# last_purchase
# days_since_last_purchase

In [0]:
# └── features
#     ├── customer_features
#     ├── product_features
#     ├── customer_product_interactions
#     └── recommendation_features

In [0]:
## customer_features
# customer_id
# customer_unique_id
# customer_state
# total_orders
# total_items
# total_spent
# avg_order_value
# avg_freight
# avg_item_price
# unique_products
# unique_categories
# favorite_category
# days_since_last_order

In [0]:
##product_features

# product_id
# product_category_name
# price
# freight_value
# product_weight_g
# product_length_cm
# product_height_cm
# product_width_cm
# total_orders
# unique_customers
# total_revenue
# avg_rating

In [0]:
# popularity
# conversion
# repeat_purchase_rate

In [0]:
# Cliente A ── Produto 1
#          ├─ Produto 2
#          └─ Produto 3

# Cliente B ── Produto 1
#          ├─ Produto 2
#          └─ Produto 4

# A → recomenda Produto 4

In [0]:
# Cliente gosta de:

# categoria = beleza
# preço = 50-100
# frete = baixo

In [0]:
# customer_id
# product_id
# customer_features
# product_features
# interaction_features
#         │
#         ▼
# Logistic Regression
#         │
#         ▼
# P(buy product | customer)



# Cliente A + Produto X → 0.82
# Cliente A + Produto Y → 0.63
# Cliente A + Produto Z → 0.12



# X
# Y